# 🚀 TMS Logistics — Algorithm Lab & Optimization Benchmark

Sổ tay này giúp bạn chạy và thử nghiệm toàn bộ hệ thống thuật toán tối ưu hóa vận tải (TMS):
- **VRP kết hợp Xếp dỡ thùng xe 2D (LIFO & Door Clearance)**
- **Tuyển chọn 10 metaheuristic** trên 12 bộ dữ liệu và nhiều random seed
- Loại Greedy và OR-Tools khỏi danh sách ứng viên; Greedy chỉ được dùng nội bộ để dựng nghiệm ban đầu
- Chỉ xếp hạng nghiệm vượt contract, time-window và production SpatialValidator
- Xuất báo cáo Markdown, JSON và CSV để chọn thuật toán tiếp tục tối ưu

---

### 💡 Hướng dẫn tăng tốc trên Google Colab:
1. Vào menu **Runtime** (Thời gian chạy) -> **Change runtime type** (Thay đổi loại thời gian chạy).
2. Chọn **CPU High-RAM**. Các thuật toán hiện tại chạy trên CPU; bật GPU không tự làm benchmark nhanh hơn.
3. Bấm **Save** (Lưu).


## 💻 1. Kiểm tra cấu hình phần cứng Google Colab
Chạy cell dưới đây để xem thông tin CPU, RAM và GPU của máy ảo:

In [ ]:
import os
import platform
import psutil

print('=' * 65)
print('💻 THÔNG TIN PHẦN CỨNG MÁY ẢO GOOGLE COLAB')
print('=' * 65)
print(f'Hệ điều hành: {platform.system()} {platform.release()}')
print(f'Python Version: {platform.python_version()}')
print(f'Số lượng vCPUs khả dụng: {os.cpu_count()}')
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f'Dung lượng RAM: {ram_gb:.2f} GB')

try:
    import subprocess
    gpu_info = subprocess.check_output('nvidia-smi', shell=True).decode()
    print('\n🎮 GPU ĐƯỢC KÍCH HOẠT:')
    for line in gpu_info.split('\n')[7:11]:
        print(' ', line)
except Exception:
    print('\nℹ️ Chế độ CPU Runtime (rất tối ưu cho đa tiến trình Python / OR-Tools)')


## 📦 2. Cài đặt các thư viện phụ thuộc (Dependencies)
Cài đặt các thư viện cho validator, kiểm thử và phân tích kết quả.

In [ ]:
!pip install --quiet numpy fastapi pydantic uvicorn pytest httpx tabulate matplotlib seaborn pandas psutil
print('✅ Đã cài dependencies cho benchmark metaheuristic')


## 📂 3. Đồng bộ mã nguồn mới nhất từ GitHub
Cell dưới đây luôn checkout đúng branch benchmark. **Không tải ZIP từ máy lên**, vì ZIP cũ có thể làm Colab chạy nhầm danh sách thuật toán.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Bang334/Project-Logictists.git"
REPO_BRANCH = "feat/phase-1-core-tms"
REPO_DIR = Path('/content/project-logictists-metaheuristics')

if (REPO_DIR / '.git').is_dir():
    subprocess.run(
        ['git', '-C', str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_BRANCH],
        check=True,
    )
    subprocess.run(
        ['git', '-C', str(REPO_DIR), 'checkout', '--force', '-B', REPO_BRANCH, 'FETCH_HEAD'],
        check=True,
    )
elif REPO_DIR.exists():
    raise RuntimeError(f'{REPO_DIR} đã tồn tại nhưng không phải Git repository. Hãy khởi động lại runtime.')
else:
    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

WORKSPACE_DIR = str(REPO_DIR)
SOURCE_COMMIT = subprocess.run(
    ['git', '-C', WORKSPACE_DIR, 'rev-parse', 'HEAD'],
    check=True, capture_output=True, text=True,
).stdout.strip()
os.chdir(WORKSPACE_DIR)
print(f'✅ Đã đồng bộ {REPO_BRANCH} tại commit {SOURCE_COMMIT}')


### 3.1. Nguồn đang sử dụng
Notebook dùng trực tiếp checkout Git ở trên; không cần chọn hoặc tải file từ máy.

In [ ]:
print(f'Repository: {REPO_URL}')
print(f'Branch: {REPO_BRANCH}')
print(f'Commit: {SOURCE_COMMIT}')


### 3.2. Kiểm tra file benchmark trong checkout

In [ ]:
required_files = [
    REPO_DIR / 'algo_lab' / 'run_parallel_benchmark.py',
    REPO_DIR / 'optimizer' / 'app' / 'spatial_validator.py',
]
missing_files = [str(path) for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError('Thiếu file bắt buộc: ' + ', '.join(missing_files))
print('✅ Checkout có đủ benchmark và SpatialValidator')


### 3.3. Làm sạch module cache, cấu hình PYTHONPATH và kiểm tra nguồn import

In [ ]:
import importlib

for module_name in list(sys.modules):
    if module_name in {'algo_lab', 'optimizer'} or module_name.startswith(('algo_lab.', 'optimizer.')):
        del sys.modules[module_name]
importlib.invalidate_caches()

for p in [WORKSPACE_DIR, os.path.join(WORKSPACE_DIR, 'optimizer'), os.path.join(WORKSPACE_DIR, 'algo_lab')]:
    if p in sys.path:
        sys.path.remove(p)
    sys.path.insert(0, p)

from algo_lab.common.models import OptimizationSolution
from algo_lab.common.data_loader import load_dataset
from optimizer.app.spatial_validator import SpatialValidator

loaded_algo_lab = Path(sys.modules['algo_lab'].__file__).resolve().parent
expected_algo_lab = (REPO_DIR / 'algo_lab').resolve()
if loaded_algo_lab != expected_algo_lab:
    raise RuntimeError(f'Import nhầm algo_lab từ {loaded_algo_lab}; cần dùng {expected_algo_lab}')
print(f'✅ Module được nạp từ commit {SOURCE_COMMIT}: {loaded_algo_lab}')


## 🧪 4. Chạy kiểm tra nhanh hệ thống (Pytest Sanity Check)
Xác nhận các quy tắc nghiệp vụ TMS: Non-stackable packing, Door clearance, Time window, Costing...

In [ ]:
!pytest -q {WORKSPACE_DIR}/algo_lab/tests/


## ⚙️ 5. Cấu hình phép thử
Mỗi metaheuristic nhận cùng ngân sách tìm kiếm. Ba seed mặc định giúp đo độ ổn định, thay vì kết luận từ một lần chạy may mắn.

In [ ]:
SEARCH_BUDGET = 10.0
SPATIAL_AUDIT_BUDGET = 5.0
REPAIR_BUDGET = 8.0
SEEDS = '0,1,2'
WORKERS = max(1, min(4, os.cpu_count() or 2))
OUTPUT_DIR = f'{WORKSPACE_DIR}/algo_lab/results'

print(f'Budget={SEARCH_BUDGET}s, seeds={SEEDS}, workers={WORKERS}')


## 🔎 6. Kiểm tra danh sách ứng viên và dữ liệu
Notebook dừng ngay nếu Greedy hoặc OR-Tools vô tình quay lại danh sách xếp hạng.

In [ ]:
from algo_lab import run_parallel_benchmark as benchmark_module

ALGORITHM_NAMES = benchmark_module.ALGORITHM_NAMES
DATASETS = benchmark_module.DATASETS
benchmark_source = Path(benchmark_module.__file__).resolve()
print(f'Nguồn benchmark: {benchmark_source} | commit: {SOURCE_COMMIT}')
if 'Greedy' in ALGORITHM_NAMES or any('OR-Tools' in name for name in ALGORITHM_NAMES):
    raise RuntimeError(
        f'Đang nạp danh sách cũ từ {benchmark_source}: {ALGORITHM_NAMES}. '
        'Hãy chọn Runtime > Disconnect and delete runtime, rồi Run all.'
    )
print(f'✅ {len(ALGORITHM_NAMES)} thuật toán × {len(DATASETS)} dataset')
for index, name in enumerate(ALGORITHM_NAMES, 1):
    print(f'{index:>2}. {name}')


## 🧪 7. Chạy benchmark tuyển chọn đa tiến trình
- Chạy **10 metaheuristic**: Standard ALNS, Genetic, Hybrid ALNS, VNS, Tabu, ILS, Late Acceptance, Simulated Annealing, GRASP và Memetic Search.
- Trên **12 kịch bản**: 4 chi nhánh DB thật (Hà Nội, Đà Nẵng, Sóng Thần, Nha Trang) + 8 kịch bản thử thách gắt gao (nhiều kiện, hàng nặng, phân tán, time window chặt, đội xe hỗn hợp, tái sử dụng sàn, đa depot, phân mảnh cao).
- Mỗi thuật toán chạy trên nhiều seed; nghiệm FAIL không được dùng để so chi phí.
- Xếp hạng ưu tiên phục vụ đủ đơn và PASS ổn định, sau đó mới xét gap chi phí và thời gian.

In [ ]:
print(f'🚀 Bắt đầu benchmark với {WORKERS} workers, {SEEDS=}, budget={SEARCH_BUDGET}s/lượt')
!python {WORKSPACE_DIR}/algo_lab/run_parallel_benchmark.py --budget {SEARCH_BUDGET} --spatial-audit-budget {SPATIAL_AUDIT_BUDGET} --repair-budget {REPAIR_BUDGET} --seeds {SEEDS} --workers {WORKERS} --output-dir {OUTPUT_DIR}


## 📋 8. Đọc kết luận tuyển chọn
Báo cáo chỉ nêu ứng viên dẫn đầu khi thuật toán đó PASS và phục vụ đủ đơn ở mọi dataset/seed.

In [ ]:
from pathlib import Path

report_path = Path(OUTPUT_DIR) / 'benchmark_10_metaheuristics_12_datasets.md'
if not report_path.is_file():
    raise FileNotFoundError('Chưa có báo cáo. Hãy chạy bước 7 trước.')
report_text = report_path.read_text(encoding='utf-8')
print(report_text)


## 📊 9. Trực quan hóa độ tin cậy, chất lượng và thời gian
Đọc JSON chuẩn hóa để biểu diễn số lượt phục vụ đủ + PASS, gap chi phí trung bình và trung vị thời gian.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

json_path = Path(OUTPUT_DIR) / 'benchmark_10_metaheuristics_12_datasets.json'
payload = json.loads(json_path.read_text(encoding='utf-8'))
ranking = pd.DataFrame(payload['ranking'])
display(ranking[['algorithm', 'full_valid_count', 'run_count', 'mean_cost_gap_percent', 'median_time_sec', 'wins']])

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
ranking.plot.barh(x='algorithm', y='full_valid_count', ax=axes[0], legend=False, title='Số lượt đủ đơn + PASS')
ranking.plot.barh(x='algorithm', y='mean_cost_gap_percent', ax=axes[1], legend=False, title='Gap chi phí trung bình (%)')
ranking.plot.barh(x='algorithm', y='median_time_sec', ax=axes[2], legend=False, title='Trung vị thời gian (giây)')
plt.tight_layout()
plt.show()


## 🧭 10. Nguyên tắc đưa thuật toán vào project
Không tự động thay production solver chỉ vì đứng hạng nhất ở benchmark ngắn. Ứng viên phải PASS toàn bộ, ổn định qua seed, sau đó chạy ngân sách dài và benchmark với dữ liệu vận hành đại diện.

In [ ]:
eligible = ranking[(ranking['full_valid_count'] == ranking['run_count']) & (ranking['pass_count'] == ranking['run_count'])]
if eligible.empty:
    print('⚠️ Chưa có thuật toán đủ điều kiện để đưa vào production.')
else:
    winner = eligible.iloc[0]
    print(f"✅ Ứng viên cần tối ưu tiếp: {winner['algorithm']}")
    print('Bước kế tiếp: tăng budget, thêm dữ liệu thật và phân tích từng lỗi/outlier trước khi tích hợp.')


## 💾 11. Tải toàn bộ kết quả về máy tính (Download Results)
Chạy cell dưới đây để nén thư mục `results/` và tự động tải về máy tính của bạn:

In [ ]:
import shutil
from google.colab import files

results_dir = os.path.join(WORKSPACE_DIR, 'algo_lab', 'results')
zip_path = '/content/tms_algo_results'

if os.path.exists(results_dir) and os.listdir(results_dir):
    shutil.make_archive(zip_path, 'zip', results_dir)
    print('🎉 Đang tải file tms_algo_results.zip về máy...')
    files.download(f'{zip_path}.zip')
else:
    print('⚠️ Thư mục results chưa có dữ liệu để tải về.')
